<a href="https://colab.research.google.com/github/shivamkumar359/03-rd-SEM--ASSIGNMENTS/blob/main/ML_W9_WEEK_9_ASSIGNMENT_Q1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib.backends.backend_pdf import PdfPages
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

In [3]:
# --------------------------------------------------
# STEP 1: LOAD THE DATASET
# --------------------------------------------------

iris = load_iris()

X = iris.data
y = iris.target

sample_ids = np.arange(len(y))

print("Dataset loaded successfully.")
print("Total samples:", len(y))
print("Number of classes:", len(iris.target_names))


Dataset loaded successfully.
Total samples: 150
Number of classes: 3


In [5]:


# --------------------------------------------------
# STEP 2: SHUFFLE THE DATASET
# --------------------------------------------------

random_seed = 42

rng = np.random.default_rng(random_seed)

shuffled_indices = rng.permutation(len(y))

X = X[shuffled_indices]
y = y[shuffled_indices]

sample_ids = sample_ids[shuffled_indices]

print("\nDataset shuffled successfully.")


Dataset shuffled successfully.


In [6]:
# --------------------------------------------------
# STEP 3: DIVIDE DATA INTO FIVE EQUAL FOLDS
# --------------------------------------------------

k = 5

folds = np.array_split(np.arange(len(y)), k)

print("\nDataset divided into", k, "folds.")

for i, fold in enumerate(folds):
    print(f"Fold {i + 1}: {len(fold)} samples")


Dataset divided into 5 folds.
Fold 1: 30 samples
Fold 2: 30 samples
Fold 3: 30 samples
Fold 4: 30 samples
Fold 5: 30 samples


In [7]:
# --------------------------------------------------
# STEP 4: SAVE DATASET SPLIT INTO CSV
# --------------------------------------------------

split_data = []

for fold_number, fold_indices in enumerate(folds, start=1):

    for index in fold_indices:

        split_data.append({
            "Sample_ID": int(sample_ids[index]),
            "Fold": fold_number,
            "Class": iris.target_names[y[index]]
        })

df = pd.DataFrame(split_data)

df.to_csv("dataset_split.csv", index=False)

print("\nDataset split saved as dataset_split.csv")


Dataset split saved as dataset_split.csv


In [8]:
# --------------------------------------------------
# STEP 5: PERFORM 5-FOLD CROSS-VALIDATION
# --------------------------------------------------

fold_accuracies = []

confusion_matrices = []

print("\n====================================")
print("       5-FOLD CROSS-VALIDATION")
print("====================================")

for fold_number in range(k):

    # Select validation fold
    validation_indices = folds[fold_number]

    # Select remaining folds for training
    training_indices = np.concatenate([
        folds[i] for i in range(k)
        if i != fold_number
    ])

    X_train = X[training_indices]
    y_train = y[training_indices]

    X_test = X[validation_indices]
    y_test = y[validation_indices]

    # Create machine learning model
    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1000, random_state=random_seed)
    )

    # Train model
    model.fit(X_train, y_train)

    # Predict validation data
    predictions = model.predict(X_test)

    # Calculate accuracy
    accuracy = accuracy_score(y_test, predictions)

    fold_accuracies.append(accuracy)

    # Calculate confusion matrix
    cm = confusion_matrix(
        y_test,
        predictions,
        labels=[0, 1, 2]
    )

    confusion_matrices.append(cm)

    print(
        f"Fold {fold_number + 1} Accuracy: "
        f"{accuracy:.4f} ({accuracy * 100:.2f}%)"
    )


       5-FOLD CROSS-VALIDATION
Fold 1 Accuracy: 0.9333 (93.33%)
Fold 2 Accuracy: 1.0000 (100.00%)
Fold 3 Accuracy: 0.8667 (86.67%)
Fold 4 Accuracy: 1.0000 (100.00%)
Fold 5 Accuracy: 0.9667 (96.67%)


In [15]:
# --------------------------------------------------
# STEP 6: CALCULATE FINAL AVERAGE ACCURACY
# --------------------------------------------------

average_accuracy = sum(fold_accuracies) / k

print("\n====================================")
print("          FINAL RESULTS")
print("====================================")

for i, accuracy in enumerate(fold_accuracies, start=1):
    print(f"Fold {i}: {accuracy * 100:.2f}%")

print

print("\n====================================\n"
    f"Average Accuracy: "
    f"{average_accuracy * 100:.2f}%"
)

print("====================================")


          FINAL RESULTS
Fold 1: 93.33%
Fold 2: 100.00%
Fold 3: 86.67%
Fold 4: 100.00%
Fold 5: 96.67%

Average Accuracy: 95.33%


In [16]:

# --------------------------------------------------
# STEP 7: GENERATE PDF CONFUSION MATRICES
# --------------------------------------------------

with PdfPages("validation_results.pdf") as pdf:

    for fold_number, cm in enumerate(
        confusion_matrices, start=1
    ):

        fig, ax = plt.subplots(figsize=(7, 5))

        display = ConfusionMatrixDisplay(
            confusion_matrix=cm,
            display_labels=iris.target_names
        )

        display.plot(
            ax=ax,
            cmap="Blues",
            colorbar=False
        )

        ax.set_title(
            f"Confusion Matrix - Fold {fold_number}"
        )

        plt.tight_layout()

        pdf.savefig(fig)

        plt.close(fig)


print("\nPDF report saved as validation_results.pdf")


PDF report saved as validation_results.pdf


In [17]:
# --------------------------------------------------
# STEP 8: FINAL SUMMARY
# --------------------------------------------------

print("\n====================================")
print("       GENERATED DELIVERABLES")
print("====================================")

print("1. kfold_implementation.py")
print("2. dataset_split.csv")
print("3. validation_results.pdf")
print("4. Terminal output screenshot")

print("\n5-Fold Cross-Validation completed successfully!")


       GENERATED DELIVERABLES
1. kfold_implementation.py
2. dataset_split.csv
3. validation_results.pdf
4. Terminal output screenshot

5-Fold Cross-Validation completed successfully!
